# S2 · Conocimiento y memoria
Tres tipos de contexto para tu agente: **conocimiento externo** (la web y fuentes abiertas), **conocimiento interno** (tus documentos, con RAG) y **memoria** (lo que la persona ya dijo). Cada bloque empieza con un **ejemplo mínimo** (lo que viste en la demo) y sigue con un **ejercicio** que lo varía. Son 6 ejercicios y un reto (≈ 60 min).
Corre sin LLM con el modelo simulado (`COURSE_OFFLINE=1`, por defecto). Para IA real: `COURSE_OFFLINE=0` en `.env` y *Restart* del kernel. La búsqueda web y las APIs necesitan internet.


## Preparación
`crear_modelo` te da el modelo de tu ruta y `crear_embeddings` el **modelo de embeddings** que usará Chroma para buscar por significado: `embeddinggemma` con Ollama (`ollama pull embeddinggemma`), `gemini-embedding-001` con Gemini, o el modelo pequeño que trae Chroma en el modo simulado (se descarga una vez, ~80 MB).


In [ ]:
from pathlib import Path

import chromadb
import httpx
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic_ai import Agent, ConversationTypeAdapter
from pydantic_ai.capabilities import Capability, WebFetch, WebSearch
from pydantic_ai.common_tools.web_fetch import web_fetch_tool

from utilidades.config import crear_embeddings, crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar, ver_mensajes

settings, OFFLINE = preparar()
modelo = crear_modelo(settings, OFFLINE)
embeddings = crear_embeddings(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE))

NOTAS = Path("datos/notas")  # notas ficticias de Alex
RECETAS = Path("datos/recetas")  # recetas ficticias para tus ejercicios


def tools_usadas(resultado) -> set[str]:
    """Nombres de las tools que el modelo pidió en un run (para autocorregir)."""
    return {
        p.tool_name
        for m in resultado.all_messages()
        for p in m.parts
        if p.part_kind == "tool-call" and p.tool_name != "final_result"
    }


## 1 · Conocimiento externo
## Ejemplo 1 · Buscar en la web (Demo 1)
`WebSearch` es una **capability ya hecha** de Pydantic AI: no escribes la tool, la enchufas. Con `native=False, local=True` busca siempre con **DuckDuckGo** (gratis, con cualquier modelo; el buscador propio de Gemini no entra en la capa gratuita).


In [ ]:
buscador = Agent(
    modelo,
    instructions="Responde en español, breve. Para datos actuales busca en la web y di de qué URL sacaste el dato.",
    capabilities=[WebSearch(native=False, local=True)],
)
r = await buscador.run("Busca en la web qué es Pydantic AI.")
print(r.output)
ver_mensajes(r)


## Ejercicio 1 · Leer una página
**Se apoya en:** Demo 1 · **Nivel:** básico · ≈ 5 min
Dale al agente `lector` la capability `WebFetch`, que descarga una URL y la convierte en texto: `WebFetch(native=False, local=web_fetch_tool(max_content_length=5000))`. El recorte a 5000 caracteres importa: el modelo tiene una **ventana de contexto** limitada y una página entera no cabe.


In [ ]:
# TODO Ejercicio 1: añade capabilities=[WebFetch(native=False, local=web_fetch_tool(max_content_length=5000))]
lector = Agent(modelo, instructions="Responde en español, en una o dos frases.")
r = await lector.run("Lee la página https://es.wikipedia.org/wiki/Quillacollo y dime en qué departamento está.")
print(r.output)
ver_mensajes(r)
revisar("Ejercicio 1 · el modelo pidió web_fetch", lambda: "web_fetch" in tools_usadas(r), "¿añadiste WebFetch a capabilities?")


## Ejemplo 2 · Una fuente de confianza (Demo 2)
La web abierta mezcla de todo. Para un tema concreto, una **tool tuya** que consulta una API de confianza controla la fuente y devuelve **solo** lo útil, incluida la **URL para citar**. Es lo mismo que en la S1 (`httpx` dentro de una tool), ahora como conocimiento.


In [ ]:
enciclopedia = Capability(
    id="enciclopedia",
    description="Resúmenes de Wikipedia en español.",
    instructions="Para definiciones usa resumen_wikipedia. Termina con «Fuente: <url>».",
)


@enciclopedia.tool_plain
async def resumen_wikipedia(tema: str) -> dict:
    """Resumen corto de un artículo de Wikipedia en español, con su URL. Ejemplo de tema: «Cochabamba»."""
    titulo = tema[:1].upper() + tema[1:].replace(" ", "_")
    url = f"https://es.wikipedia.org/api/rest_v1/page/summary/{titulo}"
    async with httpx.AsyncClient(timeout=15, headers={"User-Agent": "curso-agentes"}) as cliente:
        respuesta = await cliente.get(url, follow_redirects=True)
    if respuesta.status_code != 200:
        return {"error": f"No encontré «{tema}» en Wikipedia"}
    datos = respuesta.json()
    return {"titulo": datos["title"], "resumen": datos["extract"], "url": datos["content_urls"]["desktop"]["page"]}


print(await resumen_wikipedia("Cochabamba"))  # la función sola, sin modelo


## Ejercicio 2 · Tu fuente: libros de Open Library
**Se apoya en:** Demo 2 · **Nivel:** medio · ≈ 12 min
Escribe en la Capability `libros_abiertos` la tool `async def buscar_libro(titulo: str) -> dict`:
1. Haz `GET https://openlibrary.org/search.json` con `params={"q": titulo, "limit": 1}` (API abierta, sin clave).
2. Toma `respuesta.json()["docs"]`. Si está vacío, devuelve `{"error": "..."}`.
3. Si no, con el primer resultado `doc` devuelve `{"titulo": doc["title"], "autor": doc["author_name"][0], "anio": doc["first_publish_year"], "url": "https://openlibrary.org" + doc["key"]}`.


In [ ]:
libros_abiertos = Capability(
    id="libros_abiertos",
    description="Datos de libros del catálogo abierto Open Library.",
    instructions="Para preguntas sobre libros usa buscar_libro y cita la url.",
)

# TODO Ejercicio 2: decora con @libros_abiertos.tool_plain una función async buscar_libro(titulo: str) -> dict
# (mira resumen_wikipedia: misma forma, otra URL y otros campos).
try:  # primero la función sola, sin modelo
    prueba, desconocido = await buscar_libro("Raza de bronce"), await buscar_libro("zzqxw libro inexistente")
except NameError:
    prueba = desconocido = None
print(prueba, desconocido)
revisar("Ejercicio 2 · devuelve autor y url", lambda: "Arguedas" in prueba["autor"] and prueba["url"].startswith("https://openlibrary.org"), "devuelve autor y url")
revisar("Ejercicio 2 · libro inexistente → error amable", lambda: "error" in desconocido, "si docs está vacío, devuelve {'error': ...}")
bibliotecario = Agent(modelo, instructions="Responde en español, breve.", capabilities=[libros_abiertos])
r = await bibliotecario.run("¿Quién escribió el libro Juan de la Rosa y en qué año se publicó?")
print(r.output)
revisar("Ejercicio 2 · el modelo pidió buscar_libro", lambda: "buscar_libro" in tools_usadas(r), "¿la tool está en la Capability libros_abiertos?")


## 2 · Conocimiento interno (RAG)
## Ejemplo 3 · Fragmentar e indexar tus notas (Demo 3)
**RAG** = *Retrieval-Augmented Generation*: antes de responder, el agente **recupera** trozos de tus documentos. Dos pasos que se hacen una vez:
1. **Fragmentar** (`RecursiveCharacterTextSplitter`): trozos de ~300 caracteres que se solapan, cortando de preferencia entre párrafos.
2. **Indexar** (Chroma): guarda cada trozo con su **embedding** (un vector que representa su significado) y su `id`, que usaremos como **fuente**.

Después, `query` busca por **significado**, no por palabras iguales.


In [ ]:
divisor = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
cliente = chromadb.EphemeralClient()  # en memoria: se rehace cada vez que corres el notebook
notas = cliente.get_or_create_collection("notas", embedding_function=embeddings)

for archivo in sorted(NOTAS.glob("*.md")):
    trozos = divisor.split_text(archivo.read_text(encoding="utf-8"))
    notas.upsert(ids=[f"{archivo.name}#{i}" for i in range(len(trozos))], documents=trozos)

print(notas.count(), "fragmentos")
r = notas.query(query_texts=["¿Cuánta plata necesito para la excursión?"], n_results=2)
print(r["ids"][0], r["distances"][0])  # distancia: más chica = más parecido


## Ejercicio 3 · El tamaño del fragmento
**Se apoya en:** Demo 3 · **Nivel:** básico · ≈ 5 min
Crea `divisor_chico` con `chunk_size=120` y `chunk_overlap=20`, y fragmenta con él la nota `viajes.md` en `trozos_chicos`. Compara: ¿cuántos trozos salen con 300 y con 120? ¿Qué se pierde con trozos muy chicos? ¿Y con trozos muy grandes?


In [ ]:
texto_viajes = (NOTAS / "viajes.md").read_text(encoding="utf-8")
trozos_grandes = divisor.split_text(texto_viajes)
# TODO Ejercicio 3: crea divisor_chico y trozos_chicos.
divisor_chico = None
trozos_chicos = []
print(len(trozos_grandes), "trozos de 300 ·", len(trozos_chicos), "trozos de 120")
for trozo in trozos_chicos[:3]:
    print("·", trozo)
revisar("Ejercicio 3 · más trozos con chunk_size=120", lambda: len(trozos_chicos) > len(trozos_grandes), "usa divisor_chico.split_text(texto_viajes)")
revisar("Ejercicio 3 · ningún trozo pasa de 120 caracteres", lambda: trozos_chicos and max(map(len, trozos_chicos)) <= 120, "chunk_size=120")


## Ejercicio 4 · Tu propia base: un recetario
**Se apoya en:** Demo 3 · **Nivel:** medio · ≈ 10 min
En `datos/recetas/` hay cuatro recetas ficticias. Crea la colección `recetas` (con `embedding_function=embeddings`) y añade los trozos de cada receta con `upsert`, con ids como `"silpancho.md#0"`. Después, prueba preguntas que **no** usen las palabras exactas de la receta.


In [ ]:
# TODO Ejercicio 4: crea la colección `recetas` y recorre RECETAS.glob("*.md") como en el ejemplo 3.
recetas = None
if recetas is not None:
    for pregunta in ["Quiero algo sin carne para almorzar", "¿Qué plato lleva huevo frito?"]:
        r = recetas.query(query_texts=[pregunta], n_results=1)
        print(pregunta, "→", r["ids"][0])
revisar("Ejercicio 4 · las cuatro recetas están indexadas", lambda: len({i.split("#")[0] for i in recetas.get()["ids"]}) == 4, "recorre RECETAS.glob('*.md') y usa ids 'archivo#i'")
revisar(
    "Ejercicio 4 · «sin carne» encuentra la ensalada",
    lambda: recetas.query(query_texts=["Quiero algo sin carne para almorzar"], n_results=1)["ids"][0][0].startswith("ensalada"),
    "¿pasaste embedding_function=embeddings?",
)


## Ejemplo 4 · RAG agéntico: buscar es una tool (Demo 4)
En vez de buscar **siempre** antes de responder, le damos al agente una tool de búsqueda: **el modelo decide** cuándo buscar y con qué palabras. La tool devuelve cada trozo con su **fuente**, y las instrucciones piden citarla y admitir cuando no está.


In [ ]:
mis_notas = Capability(
    id="mis_notas",
    description="Notas personales de Alex: preferencias, viajes, plan de aprendizaje.",
    instructions=(
        "Para datos personales usa buscar_notas y cita la fuente entre corchetes, por ejemplo [viajes.md#0]. "
        "Si las notas no tienen la respuesta, di «No lo encuentro en tus notas»."
    ),
)


@mis_notas.tool_plain
def buscar_notas(pregunta: str) -> list[dict]:
    """Busca en las notas personales de Alex los fragmentos más parecidos a la pregunta."""
    r = notas.query(query_texts=[pregunta], n_results=3)
    return [{"fuente": f, "texto": t} for f, t in zip(r["ids"][0], r["documents"][0])]


asistente = Agent(modelo, instructions="Eres el asistente de Alex: quien te escribe es Alex. Responde breve.", capabilities=[mis_notas])
r = await asistente.run("Busca en mis notas cuánto dinero necesito para el viaje a Tarata.")
print(r.output)
ver_mensajes(r)


## Ejercicio 5 · Un asistente de cocina con citas
**Se apoya en:** Demo 4 · **Nivel:** medio · ≈ 10 min
Escribe en la Capability `recetario` la tool `buscar_recetas(pregunta: str) -> list[dict]` que consulte tu colección `recetas` (3 resultados) y devuelva `[{"fuente": ..., "texto": ...}]`. Después pregunta al `cocinero` algo que esté en las recetas y algo que **no** esté.


In [ ]:
recetario = Capability(
    id="recetario",
    description="Recetas guardadas: ingredientes y preparación.",
    instructions=(
        "Para preguntas de cocina usa buscar_recetas y cita la fuente entre corchetes. "
        "Si las recetas no tienen la respuesta, dilo. No inventes recetas."
    ),
)

# TODO Ejercicio 5: decora con @recetario.tool_plain una función buscar_recetas(pregunta: str) -> list[dict]
# (mira buscar_notas: misma forma, otra colección).
try:
    prueba = buscar_recetas("bebida caliente de maíz")
except NameError:
    prueba = []
print(prueba[:1])
revisar("Ejercicio 5 · buscar_recetas devuelve fuente y texto", lambda: {"fuente", "texto"} <= set(prueba[0]), "cada elemento: {'fuente': id, 'texto': documento}")
cocinero = Agent(modelo, instructions="Eres un asistente de cocina. Responde en español, breve.", capabilities=[recetario])
r = await cocinero.run("Busca en las recetas qué ingredientes lleva el api morado.")
print(r.output)
ver_mensajes(r)
revisar("Ejercicio 5 · el modelo pidió buscar_recetas", lambda: "buscar_recetas" in tools_usadas(r), "¿la tool está en la Capability recetario?")


## 3 · Memoria
## Ejemplo 5 · La memoria de la conversación (Demo 5)
El modelo **no recuerda nada** entre llamadas: cada `run` empieza de cero. Para continuar una conversación se le pasa la anterior con `conversation=r.conversation` (memoria de **corto plazo**).


In [ ]:
charlador = Agent(modelo, instructions="Eres un asistente personal. Responde en español, breve.")
t1 = await charlador.run("Me llamo Ana y el sábado viajo a Tarata.")
t2 = await charlador.run("¿A dónde viajo el sábado?", conversation=t1.conversation)
print(t2.output)
print(len(t2.all_messages()), "mensajes en la conversación")


## Ejercicio 6 · Una memoria que sobrevive al reinicio
**Se apoya en:** Demo 5 · **Nivel:** medio · ≈ 10 min
Si el programa se cierra, `t2.conversation` se pierde. Completa:
- `guardar(persona, conversacion)`: escribe `ConversationTypeAdapter.dump_json(conversacion)` en `conversaciones/<persona>.json`.
- `cargar(persona)`: si el archivo existe, devuelve `ConversationTypeAdapter.validate_json(archivo.read_bytes())`; si no, `None` (una persona nueva empieza de cero).


In [ ]:
CONVERSACIONES = Path("conversaciones")
CONVERSACIONES.mkdir(exist_ok=True)

# TODO Ejercicio 6: completa guardar y cargar.


def guardar(persona: str, conversacion) -> None:
    pass


def cargar(persona: str):
    return None


guardar("ana", t2.conversation)
recuperada = cargar("ana")
revisar("Ejercicio 6 · se guarda y se recupera", lambda: len(recuperada.messages) == len(t2.all_messages()), "escribe y lee conversaciones/ana.json")
revisar("Ejercicio 6 · una persona nueva empieza de cero", lambda: recuperada is not None and cargar("persona-nueva") is None, "si el archivo no existe, devuelve None")
t3 = await charlador.run("¿Cómo me llamo?", conversation=recuperada)  # como si el programa hubiera reiniciado
print(t3.output)


## Reto (opcional) · Recordar y olvidar
**Se apoya en:** Demo 6 · **Nivel:** reto
La memoria de **largo plazo** guarda datos duraderos de la persona y los trae a **cada** conversación. Crea la Capability `memoria` con:
1. La tool `recordar(dato: str) -> str`, que añade el dato como una línea de `recuerdos.txt` (modo `"a"`).
2. La tool `olvidar(dato: str) -> str`, que reescribe el archivo sin las líneas que contengan `dato`.
3. Unas instrucciones **dinámicas**: una función `lo_que_se() -> str` en `instructions=[..., lo_que_se]`, que devuelva los recuerdos actuales. Pruébalo: «Recuerda que soy vegetariana», y en **otro** run sin historial «¿Qué almuerzo hoy?».


In [ ]:
RECUERDOS = Path("recuerdos.txt")
RECUERDOS.unlink(missing_ok=True)


def leer_recuerdos() -> list[str]:
    return RECUERDOS.read_text(encoding="utf-8").splitlines() if RECUERDOS.exists() else []


# TODO Reto (opcional): escribe lo_que_se, la Capability memoria y las tools recordar y olvidar.
